<a href="https://colab.research.google.com/github/chhote2827-ctrl/E-Commerce-Sales-Analysis-Dashboard/blob/main/Features_Engineering_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer

# -------------------------------------------------------------
# STEP 1: Creating a Sample Raw Dataset (Mock E-commerce/User Data)
# -------------------------------------------------------------
df = pd.DataFrame({
    "Age": [22, 30, np.nan, 45, 50, 35],
    "Salary": [30000, 50000, 60000, np.nan, 120000, 450000], # 450000 is an outlier
    "Department": ["HR", "Tech", "Finance", "Tech", "HR", "Finance"],
    "Approved": [0, 1, 1, 1, 0, 1]
})

print("--- Original Raw Data ---")
print(df)

# -------------------------------------------------------------
# STEP 2: Handling Missing Values (Using Median & KNN Imputer)
# -------------------------------------------------------------
# Age ke missing values ko median se fill karna[cite: 6]
df["Age"] = df["Age"].fillna(df["Age"].median())

# Salary ke missing values ke liye KNN Imputer ka use karna[cite: 8]
imputer = KNNImputer(n_neighbors=2)
df[["Age", "Salary"]] = imputer.fit_transform(df[["Age", "Salary"]])


# -------------------------------------------------------------
# STEP 3: Outlier Detection & Handling (Using IQR Method)
# -------------------------------------------------------------
Q1 = df["Salary"].quantile(0.25)
Q3 = df["Salary"].quantile(0.75)
IQR = Q3 - Q1

# Upper limit se badi values (Outliers) ko cap/filter karna[cite: 24]
upper_limit = Q3 + 1.5 * IQR
df["Salary"] = np.where(df["Salary"] > upper_limit, upper_limit, df["Salary"])


# -------------------------------------------------------------
# STEP 4: Categorical Encoding (Converting Text to Numbers)
# -------------------------------------------------------------
# Department text column ko Label Encoder se numbers me badalna[cite: 13]
encoder = LabelEncoder()
df["Department_Encoded"] = encoder.fit_transform(df["Department"])

# Original text column ko drop kar dena
df = df.drop(columns=["Department"])


# -------------------------------------------------------------
# STEP 5: Train-Test Split & Feature Scaling (Preventing Data Leakage)
# -------------------------------------------------------------
# X (Features) aur y (Target) alag karna[cite: 31]
X = df.drop(columns=["Approved"])
y = df["Approved"]

# Data ko 75% training aur 25% testing me split karna[cite: 31]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Standard Scaling (Data ko mean=0 aur std=1 par scale karna)[cite: 17]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Processed & Scaled Training Data Shape ---")
print(X_train_scaled.shape)

--- Original Raw Data ---
    Age    Salary Department  Approved
0  22.0   30000.0         HR         0
1  30.0   50000.0       Tech         1
2   NaN   60000.0    Finance         1
3  45.0       NaN       Tech         1
4  50.0  120000.0         HR         0
5  35.0  450000.0    Finance         1

--- Processed & Scaled Training Data Shape ---
(4, 3)
